# NetSciX 2027 · Notebook 3 — Structural analysis and sensitivity of spatial networks

**Workshop:** *From Incomplete Spatial Data to Networks: A Hands-on Tutorial in Predictive Modeling and Routing*  
**Organizers:** Javier M. Vera Zúñiga & Dina S. Cornejo Meza

**Question:** How much of a modeled network's structure depends on the assumed route costs?

This notebook analyses the **same candidate nodes and endpoint pairs** under two path-cost scenarios created by Notebook 2. It distinguishes:

1. **Unweighted topology:** node and link identity, components, degree and unweighted betweenness. These are **identical by construction** when endpoints are held fixed.
2. **Length-weighted structure:** shortest paths on the candidate-node graph, where each link is weighted by its modeled **physical route length**. These measures *can* change as route geometry changes.
3. **Sensitivity to an explicit edge-retention rule:** optional filtering by route detour ratio, which *can* change the graph's topology but introduces another modeling assumption.

**Scientific and privacy limits:** candidate nodes are neither documented archaeological sites nor the 24 unsupervised attractors of the original Cañete corridor model. Route lengths are computational outputs, not measured historic pathways. In **real** mode, all node coordinates, route endpoints, and derived results are heritage-sensitive and must not be published without clearance. No private exports are generated automatically.

## 0. Mode and dependencies

- The default is `DATA_MODE="demo"`; keep it to run independently, with an **embedded synthetic output** generated by the v0.3 Notebook 2 tutorial. No files or external repository needed.
- Choose `DATA_MODE="real"` to upload **two private, prepared NB2 outputs**: `nb2_nodes_PRIVATE.csv` and `nb2_edges_PRIVATE.csv` from `02_PRIVADO_NO_SUBIR/data/` in the unified workshop ZIP. The optional route-cell file is not required.
- Rerunning Notebook 2 with `EXPORT_RESULTS=True` creates the same type of private handoff. Do not upload or commit the private outputs to a public GitHub repository.

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from scipy.stats import spearmanr

DATA_MODE = os.getenv("NETSCIX_DATA_MODE", "demo")  # "real" or "demo"
EXPORT_RESULTS = False  # Keep False in real mode unless authorized private export is required.
assert DATA_MODE in ("demo", "real")
print("Mode:", DATA_MODE, "| exports enabled:", EXPORT_RESULTS)

## 1. Load the network handoff, independently of the other Colab sessions

In `demo` mode, the small embedded dataset is derived from the **synthetic NB2 run** and has six computational nodes and seven candidate pairs. In `real` mode, files from the private package contain Cañete-derived results (twelve candidate nodes and seventeen pairs in the tested v0.3 run). The same candidate pairs must occur in **both** scenarios.

In [ ]:
# Embedded public synthetic NB2 outputs (no real archaeological locations)
DEMO_NODES = [('N01', 5250.0, 3250.0), ('N02', 1250.0, 3750.0), ('N03', 7000.0, 6000.0), ('N04', 5750.0, 0.0), ('N05', 1500.0, 0.0), ('N06', 3000.0, 6250.0)]
DEMO_EDGES = [('slope_dominant', 'N01', 'N03', 3767.766953, 7951.912755), ('slope_dominant', 'N01', 'N04', 3457.106781, 6757.919702), ('slope_dominant', 'N01', 'N06', 4664.213562, 8994.795646), ('slope_dominant', 'N02', 'N05', 3853.553391, 7913.62097), ('slope_dominant', 'N02', 'N06', 3224.873734, 7070.080745), ('slope_dominant', 'N03', 'N06', 4103.553391, 11618.168164), ('slope_dominant', 'N04', 'N05', 4457.106781, 12721.757119), ('river_distance_dominant', 'N01', 'N03', 3621.320344, 7907.831027), ('river_distance_dominant', 'N01', 'N04', 3457.106781, 6757.69286), ('river_distance_dominant', 'N01', 'N06', 4664.213562, 8924.944133), ('river_distance_dominant', 'N02', 'N05', 3853.553391, 9056.882494), ('river_distance_dominant', 'N02', 'N06', 3224.873734, 6890.473531), ('river_distance_dominant', 'N03', 'N06', 4103.553391, 13802.649575), ('river_distance_dominant', 'N04', 'N05', 4250.0, 14192.170497)]

def find_private(name):
    extra = os.getenv("NETSCIX_PRIVATE_DIR", "")
    folders = ([Path(extra)] if extra else []) + [Path('/content'), Path.cwd(), Path('data/private'), Path('../data/private')]
    return next((p/name for p in folders if (p/name).is_file()), None)

if DATA_MODE == 'demo':
    nodes = pd.DataFrame(DEMO_NODES,columns=['node_id','x','y'])
    edges = pd.DataFrame(DEMO_EDGES,columns=['scenario','source','target','physical_length_m','accumulated_model_cost'])
    print('SYNTHETIC network from Notebook 2 demo — no archaeological data.')
else:
    names=['nb2_nodes_PRIVATE.csv','nb2_edges_PRIVATE.csv']
    missing=[name for name in names if find_private(name) is None]
    if missing:
        try:
            from google.colab import files
            print('Upload only these TWO private NB2 files:',missing)
            files.upload()
        except ImportError as exc:
            raise FileNotFoundError('Files not found. Set NETSCIX_PRIVATE_DIR or choose demo mode.') from exc
    paths={name:find_private(name) for name in names}
    if any(v is None for v in paths.values()):
        raise FileNotFoundError('Missing private NB2 files. Check uploaded filenames.')
    nodes=pd.read_csv(paths[names[0]])
    edges=pd.read_csv(paths[names[1]])
    print('PRIVATE Cañete NB2 results loaded. Do NOT publish these results or maps.')

required_nodes={'node_id','x','y'}
required_edges={'scenario','source','target','physical_length_m','accumulated_model_cost'}
assert required_nodes.issubset(nodes.columns) and required_edges.issubset(edges.columns)
assert nodes.node_id.is_unique and len(nodes)>2
assert nodes[['x','y']].notna().all().all()
assert np.isfinite(edges[['physical_length_m','accumulated_model_cost']].to_numpy(dtype=float)).all()
assert (edges[['physical_length_m','accumulated_model_cost']] > 0).all().all()
assert (edges.source != edges.target).all()
assert set(edges.source).union(edges.target).issubset(set(nodes.node_id))
scenarios=sorted(edges.scenario.unique())
assert len(scenarios)==2, 'Expect exactly two routing scenarios.'
assert not edges.duplicated(['scenario','source','target']).any()

def canonical_pairs(part):
    return {tuple(sorted((a,b))) for a,b in part[['source','target']].itertuples(index=False,name=None)}
pairsets=[canonical_pairs(edges[edges.scenario==scenario]) for scenario in scenarios]
assert len(pairsets[0])==len(pairsets[1]) and pairsets[0]==pairsets[1], 'Scenarios must use the same endpoint pairs.'
assert all(len(edges[edges.scenario==s])==len(pairsets[0]) for s in scenarios)
print('Nodes:',len(nodes),'| endpoint pairs:',len(pairsets[0]),'| scenarios:',scenarios)

## 2. Construct the two candidate-node graphs

A link connects two candidates selected by the **same pairing rule** in Notebook 2. Edge `length_m` is the physical distance traveled along the modeled least-cost path, not straight-line distance. We calculate betweenness using **shortest total physical route length** at the abstract node-network level. We avoid comparing accumulated model cost between differently scaled surfaces.

In [ ]:
def make_graph(scenario):
    G=nx.Graph(scenario=scenario)
    for row in nodes.itertuples(index=False):
        G.add_node(row.node_id, x=float(row.x), y=float(row.y))
    for row in edges.loc[edges.scenario.eq(scenario)].itertuples(index=False):
        G.add_edge(row.source,row.target,length_m=float(row.physical_length_m),model_cost=float(row.accumulated_model_cost))
    return G

graphs={scenario:make_graph(scenario) for scenario in scenarios}
assert all(G.number_of_nodes()==len(nodes) and G.number_of_edges()==len(pairsets[0]) for G in graphs.values())
print('Built graphs:',[(s,G.number_of_nodes(),G.number_of_edges()) for s,G in graphs.items()])

## 3. Topology and length-sensitive metrics

- **Degree and unweighted betweenness** use just the fixed endpoint pairs; they must match between scenarios.
- **Length-weighted betweenness** can change even with fixed pairs, because a different set of physical route lengths can change shortest routes *through the candidate-node network*.
- **Weighted mean shortest-path length** is defined only for a connected graph. If a graph is disconnected, it is shown as `NaN` rather than silently excluding unreachable pairs.

In [ ]:
def metrics(s,G):
    n=G.number_of_nodes()
    cc=nx.number_connected_components(G)
    deg=dict(G.degree())
    bc_u=nx.betweenness_centrality(G,weight=None,normalized=True)
    bc_w=nx.betweenness_centrality(G,weight='length_m',normalized=True)
    shortest_mean=float(nx.average_shortest_path_length(G,weight='length_m')) if nx.is_connected(G) else np.nan
    summary=dict(scenario=s,n_nodes=n,n_edges=G.number_of_edges(),components=cc,
                 density=nx.density(G),mean_degree=float(np.mean(list(deg.values()))),
                 mean_edge_length_m=float(np.mean([d['length_m'] for _,_,d in G.edges(data=True)])),
                 mean_weighted_shortest_path_m=shortest_mean)
    central=pd.DataFrame({'node_id':list(G.nodes),'scenario':s,
             'degree':[deg[k] for k in G.nodes],
             'betweenness_unweighted':[bc_u[k] for k in G.nodes],
             'betweenness_length_weighted':[bc_w[k] for k in G.nodes]})
    return summary,central
summaries=[];centrals=[]
for scenario,G in graphs.items():
    summary,central=metrics(scenario,G);summaries.append(summary);centrals.append(central)
summary_df=pd.DataFrame(summaries)
central_df=pd.concat(centrals,ignore_index=True)
print('Network metrics (rounded):')
print(summary_df.round(4).to_string(index=False))
check=central_df.pivot(index='node_id',columns='scenario',values='betweenness_unweighted')
assert np.allclose(check.iloc[:,0],check.iloc[:,1]),'Unweighted topology must be invariant.'
assert summary_df[['n_nodes','n_edges','components','density','mean_degree']].nunique().max()==1
print('QA: topology invariance confirmed (same nodes and endpoint pairs).')

## 4. Which candidates change their *weighted* network roles?

Compute rank changes in length-weighted betweenness. A top-ranked candidate is merely **central in this modeled network**, not historically important. Ties may occur, so inspect the actual centrality values as well as ranks.

In [ ]:
A,B=scenarios
cA=central_df.query('scenario == @A').set_index('node_id')
cB=central_df.query('scenario == @B').set_index('node_id')
comparison=pd.DataFrame({
    f'betweenness_{A}':cA.betweenness_length_weighted,
    f'betweenness_{B}':cB.betweenness_length_weighted,
})
comparison['absolute_difference']=(comparison.iloc[:,0]-comparison.iloc[:,1]).abs()
comparison['rank_A']=comparison.iloc[:,0].rank(method='min',ascending=False).astype(int)
comparison['rank_B']=comparison.iloc[:,1].rank(method='min',ascending=False).astype(int)
comparison=comparison.sort_values('absolute_difference',ascending=False)
print(comparison.round(4).head(12).to_string())
rho,pvalue=spearmanr(comparison.rank_A,comparison.rank_B)
print('Spearman rank correlation:', round(float(rho),3) if np.isfinite(rho) else 'undefined (all ranks identical)',
      '| descriptive comparison only; p-value is not a validation test.')

## 5. Visualize both networks

The edges are **straight lines drawn between computational node coordinates**, with width related to modeled route length. They are **not** the actual meandering least-cost paths; those are in the separate route-cell output from Notebook 2. The coordinate axes are hidden by default. In real mode, do not share or publish these plots without heritage-data clearance.

In [ ]:
positions={r.node_id:(float(r.x),float(r.y)) for r in nodes.itertuples(index=False)}
fig,axs=plt.subplots(1,2,figsize=(12,5),constrained_layout=True)
for ax,s in zip(axs,scenarios):
    G=graphs[s]
    bc=nx.betweenness_centrality(G,weight='length_m',normalized=True)
    sizes=[120+1000*bc[v] for v in G.nodes]
    lengths=[G[u][v]['length_m'] for u,v in G.edges]
    widths=[1.0+2.0*(1-x/max(lengths)) for x in lengths]
    nx.draw_networkx_edges(G,positions,ax=ax,width=widths,alpha=.5,edge_color='tab:gray')
    nx.draw_networkx_nodes(G,positions,ax=ax,node_size=sizes,node_color='tab:blue',alpha=.8)
    nx.draw_networkx_labels(G,positions,ax=ax,font_size=8,font_color='white')
    ax.set_title(s.replace('_',' '));ax.set_aspect('equal');ax.axis('off')
fig.suptitle('SYNTHETIC modeled connections' if DATA_MODE=='demo' else 'PRIVATE modeled Cañete candidate networks — DO NOT PUBLISH')
plt.show()

## 6. Optional topological sensitivity: explicit detour filter

So far the **pair topology is fixed**, so its unweighted statistics cannot change. As a separate teaching exercise, suppose we retain a candidate edge only when its modeled path is at most **`DETOUR_LIMIT` times the straight-line distance** between its endpoints.

This additional pruning rule may disconnect networks and changes their topology. It is a **hypothetical graph-construction choice**, *not* a validated historical filter. Inspect how its effect depends on the cost scenario. Keep all the original nodes in the graph, including isolated ones.

In [ ]:
DETOUR_LIMIT = 1.45   # Try 1.2, 1.45, 1.8; fixed threshold shared across both scenarios.

def detour_filtered(G,limit):
    H=G.copy()
    ratios={}
    for u,v,d in G.edges(data=True):
        euclidean=np.hypot(H.nodes[u]['x']-H.nodes[v]['x'],H.nodes[u]['y']-H.nodes[v]['y'])
        assert euclidean>0
        detour=d['length_m']/euclidean
        assert detour >= 1-1e-8, 'Physical route cannot be shorter than Euclidean distance.'
        ratios[tuple(sorted((u,v)))]=detour
        if detour > limit: H.remove_edge(u,v)
    return H,ratios
filtered={s:detour_filtered(G,DETOUR_LIMIT)[0] for s,G in graphs.items()}
filtered_df=pd.DataFrame([{'scenario':s,'retained_edges':G.number_of_edges(),
           'components':nx.number_connected_components(G),
           'isolates':nx.number_of_isolates(G)} for s,G in filtered.items()])
print('Detour threshold:',DETOUR_LIMIT)
print(filtered_df.to_string(index=False))
assert all(G.number_of_nodes()==len(nodes) for G in filtered.values())

## 7. Interpreting the results

Discussion questions:

1. Why are degree and unweighted betweenness **exactly unchanged** across the two original scenarios?
2. Did length-weighted betweenness change? How much? Would weighted results be different if model costs rather than physical lengths were chosen, and would comparisons remain meaningful?
3. How does the **explicit detour threshold** change component counts? What new assumption did we add to make topology sensitive?
4. Could any of these computational centralities be interpreted as historical importance without independent evidence? (No: they describe a hypothesis, not documented ancient movement.)

**What the workshop produces:** three linked exercises, from spatial ranking to candidate routes to structural and sensitivity analysis. They illustrate one modeling pipeline, not an archaeological reconstruction established by field evidence.

## 8. Optional export (private by default)

The output contains node identifiers tied to the sensitive empirical handoff. To save it, explicitly set `EXPORT_RESULTS=True`, use authorized private storage, and **do not commit real-derived results to GitHub**. Exports are intentionally disabled by default.

In [ ]:
if EXPORT_RESULTS:
    suffix='PRIVATE' if DATA_MODE=='real' else 'SYNTHETIC'
    out=Path('/content') if Path('/content').exists() else Path.cwd()
    summary_df.to_csv(out/f'nb3_summary_{suffix}.csv',index=False)
    central_df.to_csv(out/f'nb3_centrality_{suffix}.csv',index=False)
    filtered_df.to_csv(out/f'nb3_detour_sensitivity_{suffix}.csv',index=False)
    print('Saved three files to',out,'—',('PRIVATE; never publish' if DATA_MODE=='real' else 'synthetic'))
else:
    print('Export disabled. Outputs are visible only in this private notebook session.')

**Try it:** Set `DETOUR_LIMIT` to 1.2 and 1.8, then compare the number of components. Which changes are due to the modelled cost surface, and which are consequences of introducing an explicit edge-filtering rule?